In [ ]:
"""
    Code to generate simulated data
    Supplemental Figure S16
"""

import os
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize_scalar

n_sim = 100
save_path = 'results/sim/data/'

os.makedirs(save_path, exist_ok=True)

effect_sz = [-1, 0, 5e-4, 5e-3]
main_effect = 0.5
pgs_effect = 0.25
target_prev = 0.05
n = 2*375000

for i in range(n_sim):
    if (i%10) == 0:
        print(f'Generating dataset: {i+1}')
    
    #age = np.random.normal(size=n)
    age = np.random.randint(40, 70, size=n)
    age_mean = age.mean()
    age = age - age_mean
    
    sex = np.tile([0, 1], n // 2)
    sex_mean = np.mean(sex)
    sex = sex - sex_mean
    pgs = np.random.normal(size=n)
    
    beta_age = np.sqrt(main_effect / np.var(age))
    beta_sex = np.sqrt(main_effect / np.var(sex))
    beta_pgs = np.sqrt(pgs_effect/ np.var(pgs))
    
    age_pgs = age * pgs
    sex_pgs = sex * pgs
    
    out = [pd.DataFrame({'sex': sex + sex_mean, 'age': age + age_mean, 'pgs': pgs})]
    
    for j, effect_sz_ in enumerate(effect_sz):
        beta_age_pgs = np.sqrt(effect_sz_ / np.var(age_pgs)) if effect_sz_ > 0 else 0
        beta_sex_pgs = np.sqrt(effect_sz_ / np.var(sex_pgs)) if effect_sz_ > 0 else 0

        lin_pred = (
            beta_age * age +
            beta_sex * sex +
            beta_pgs * pgs +
            beta_age_pgs * age_pgs +
            beta_sex_pgs * sex_pgs
        )
        
        if effect_sz_ == -1:
            lin_pred = (
                beta_pgs * pgs
            )
    
        def objective(b0):
            return (expit(b0 + lin_pred).mean() - target_prev) ** 2
    
        intercept = minimize_scalar(objective, bounds=(-10, 10), method='bounded').x
        p = expit(intercept + lin_pred)
        y_binary = (np.random.uniform(size=n) < p).astype(int)
    
        df = pd.DataFrame({
            f'y{j}_binary': y_binary,
        })
        out.append(df)
    
    out = pd.concat(out, axis=1)

# Run different models

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
out[['age_norm','sex_norm']] = scaler.fit_transform(out[['age','sex']])
out['pgs*sex'] = out['pgs'] * out['sex_norm']
out['pgs*age'] = out['pgs'] * out['age_norm']

r2_results = {}

for col in ['y0_binary', 'y1_binary', 'y2_binary', 'y3_binary']:
    y = out[col]
    r2_results[col] = {}
    for var in ['age', 'sex', 'pgs', 'pgs*sex', 'pgs*age']:
        X = out[[var]]
        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)
        r2_results[col][var] = r2_score(y, y_pred)

# See model

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

print('\n**********************\nINTERACTION\n**********************')
#model = smf.logit('y1_binary ~ age + sex + pgs + age:pgs + sex:pgs', data=out).fit()
model = smf.logit('y3_binary ~ age + sex + pgs', data=out).fit()

print(model.summary())

# Run main analysis (i.e. calc_risk_across_contexts

In [ ]:
import pandas as pd
import numpy as np
import random
import time

from options.options import Options
import util.util as util
import util.pre_process as pre
import util.bootstrap_tools as btool

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import numpy as np

import torch
from scipy.stats import norm

phen_label = 'y3_binary' # change as needed

# Function to calculate relative and absolute genetic risk within strata
def gen_results(pgs, phen, pgs_res, phen_res, top_or_null=None, quant_val=None):
    results = {}

    # Calculate observed R² between residualized PGS and phenotype
    results_ = btool.efficient_r2(pgs_res, phen_res)
    results['high_pgs'] = (pgs > pgs_emerge_cutoff).mean()  # Proportion above PGS threshold
    results['r2_obs'] = results_['r2']
    results['prev'] = phen.mean(1)  # Prevalence per phenotype vector

    # Convert observed R² to liability scale
    results['r2_liab'] = btool.effecient_r2_liab(results['r2_obs'], results['prev'])

    # Convert R² to odds ratio and absolute risk based on top quantile
    results_ = btool.effecient_r2_to_risk(results['r2_obs'], results['prev'], emerge_cutoff, quant_val)
    results['top_or'] = results_['odds_ratio']
    results['top_ar'] = results_['abs_risk']
    results['top_ar_control'] = results_['abs_risk_control']
    results['quant_val'] = np.asarray(results_['quant_val'])

    # Compute overall odds ratio
    results['or'] = btool.efficient_odds_ratio(results['r2_obs'], results['prev'])

    # If null top OR is provided, compute absolute risk under null
    if top_or_null is not None:
        odds = results['top_ar_control'] / (1 - results['top_ar_control'])
        results['top_ar_null'] = top_or_null * odds / (1 + top_or_null * odds)

    return results

# Function to compute bootstrapped distributions of risk metrics
def gen_random_results(pgs_, phen_, x, y, n_bootstrap, max_rand, var_names, top_or_null, quant_val=None):
    results = {}
    for var_name in var_names:
        results[var_name] = np.full((n_bootstrap,), np.nan)  # Initialize result arrays

    n_rand_groups = (n_bootstrap // max_rand)  # Number of bootstrap batches
    for r_group in range(n_rand_groups + 1):
        r_beg = r_group * opt.max_random
        r_end = min((r_group + 1) * max_rand, n_bootstrap)
        n_rand = r_end - r_beg

        if r_beg == r_end:
            continue

        # Sample bootstrap indices
        rand_idx = torch.randint(0, len(pgs_), size=(n_rand, len(pgs_)), dtype=torch.int32) # use torch as it is a faster way to generate random ints. Can be changed.

        # Subsample data using bootstrap indices
        pgs_rand = pgs_[rand_idx]
        phen_rand = phen_[rand_idx]
        x_rand = x[rand_idx]
        y_rand = y[rand_idx]

        # Compute risk metrics for each bootstrap sample
        results_ = gen_results(pgs_rand, phen_rand, x_rand, y_rand, top_or_null, quant_val)
        for key in results:
            results[key][r_beg:r_end] = results_[key]

    return results

opt = Options()
opt.initialize()

emerge_cutoff = .05
pgs_emerge_cutoff = norm.ppf(1-emerge_cutoff)

phen = out[phen_label]
pgs = out['pgs']
pgs = (pgs-pgs.mean())/pgs.std()
covars = out[['age', 'sex']]

# regress out covars from phenotype and pgs over the full sample
phen_res, _ = pre.lin_regress_out(covars, phen)
pgs_res, _ = pre.lin_regress_out(covars, pgs)

# Get indices of subpopulations
bin_defs = {'sex': {'Female': 0, 'Male': 1}, 
            'age': [[40, 49], [50, 59], [60, 69]]}
bounds, labels, label_map = util.create_bins_and_indices(covars, bin_defs)

util.create_folder(f'results/{phen_label}')

print('\n')
print(f'Total: {len(phen)}')
print(f'Cases: {phen.sum()}')
for sex_label in ['Female', 'Male']:
    n_sex = len(phen.loc[label_map['sex: '+sex_label]])
    n_cases = phen.loc[label_map['sex: '+sex_label]].sum()
    print(f'{sex_label} Cases: {n_cases}/{n_sex}')

for k in label_map.keys():
    print(len(label_map[k]), ': ', k,)

In [ ]:
'''
Variables stored -
    high-pgs: portion of individuals above eMERGE PGS threshold within context (range: 0-1)
    prev: prevalence of disease within context (range: 0-1)
    r2_obs: Proportion of variance in disease status explained by PGS on the observed scale within a context
    r2_liab: Proportion of variance explained in disease status by PGS on the liability scale (accounting for disease prevalence) within a context
    or: odds ratio for disease per 1 std increase in PGS within a context
    top_or: odds ratio of disease for high-PGS individuals within a context
    top_ar: absolute risk of disease for high-PGS individuals within a context
    top_ar_control: absolute risk of disease for low-PGS (i.e. not high-PGS) individuals within a context
    top_ar_null: absolute risk of disease for high-PGS individuasl using top_or from overall population
'''

vars = ['high_pgs', 'prev', 'r2_obs', 'r2_liab', 
        'or', 'top_or', 'top_ar', 'top_ar_control', 'top_ar_null']

all_results, bot_results, top_results = {}, {}, {}
pgs_, phen_, x, y = pgs.values, phen.values, pgs_res.values, phen_res.values

all_results['n'] = len(pgs_)
results_ = gen_results(pgs_[None, :], phen_[None, :], x[None, :], y[None, :], None)
all_results.update({key: results_[key].item() for key in vars if key in results_})
TOP_OR_NULL = all_results.get('top_or')

In [ ]:
# Initialize arrays
n = len(labels)

# Using dictionary instead of globals() to store arrays
results = {var: np.full((n, n), np.nan) for var in vars}
bootstrap_results = {}
n_idx = np.full((n, n), np.nan)

## Intersect analysis
for i in range(n):
    for j in range(i+1):
        idx_org = np.intersect1d(label_map[labels[i]], label_map[labels[j]])
        
        n_idx[i, j] = n_idx[j, i] = len(idx_org)

        pgs_ = pgs.loc[idx_org].values
        phen_ = phen.loc[idx_org].values
        x = pgs_res.loc[idx_org].values
        y = phen_res.loc[idx_org].values

        if phen_.sum() <= 1: # skip analysis if case count less than MIN_VAL (default 20 cases)
            continue

        results_ = gen_results(pgs_[None,:], phen_[None,:], x[None,:], y[None,:], TOP_OR_NULL)
        for key in results_:
            if key not in vars:
                continue
            results[key][i, j] = results[key][j, i] = results_[key].item()

In [ ]:
import time
start_time = time.time()

for i in range(len(bounds)-1):
    for j in range(i+1):
        bootstrap_results[(i,j)] = {'min':{}, 'max':{}}
        i_strt = bounds[i]
        i_end = bounds[i+1]
        j_strt = bounds[j]
        j_end = bounds[j+1]

        context = results['top_or'][i_strt:i_end, j_strt:j_end]

        # find indeices corresponding to min and max OR
        min_idx = np.unravel_index(np.nanargmin(context), context.shape)
        max_idx = np.unravel_index(np.nanargmax(context), context.shape)

        min_idx = (min_idx[0]+i_strt, min_idx[1]+j_strt)
        max_idx = (max_idx[0]+i_strt, max_idx[1]+j_strt)

        for extreme, idx in zip(['min', 'max'], [min_idx, max_idx]):
            i_idx = idx[0]
            j_idx = idx[1]
            
            idx_org = np.intersect1d(label_map[labels[i_idx]], label_map[labels[j_idx]])
            if len(idx_org) == 0:
                continue

            pgs_ = pgs.loc[idx_org].values
            phen_ = phen.loc[idx_org].values
            x = pgs_res.loc[idx_org].values
            y = phen_res.loc[idx_org].values

            results_rand_ = gen_random_results(pgs_, phen_, x, y, opt.n_bootstrap, opt.max_random, vars, TOP_OR_NULL, None)
            
            bootstrap_results[(i,j)][extreme] = {'index': (i_idx, j_idx), 'bootstrap': {}}
            for key in results_rand_:
                bootstrap_results[(i,j)][extreme]['bootstrap'][key] = results_rand_[key]
    print(f"{i}: - time: {time.time() - start_time} seconds")
 
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")

In [ ]:
def get_pdiff(var_min, var_max):
    var_range = (var_max-var_min)
    var_pdiff = 100*((var_max/var_min)-1)

    return var_range, var_pdiff

var_range = np.full((2,2,opt.n_bootstrap+1), np.nan)
var_pdiff = np.full((2,2,opt.n_bootstrap+1), np.nan)
for i in range(len(bounds)-1):
    for j in range(i+1):
        max_idx = bootstrap_results[(i,j)]['max']['index']
        min_idx = bootstrap_results[(i,j)]['min']['index']

        max_odds = results['top_or'][max_idx]
        min_odds = results['top_or'][min_idx]
        
        max_bootstrap = bootstrap_results[(i,j)]['max']['bootstrap']['top_or']
        min_bootstrap = bootstrap_results[(i,j)]['min']['bootstrap']['top_or']

        var_range_, var_pdiff_ = get_pdiff(min_odds, max_odds)
        var_range_bootstrap, var_pdiff_bootstrap = get_pdiff(min_bootstrap, max_bootstrap)

        var_range[i,j,0] = var_range[j,i,0] =  var_range_
        var_pdiff[i,j,0] = var_pdiff[j,i,0] = var_pdiff_
        
        var_range[i,j,1:] = var_range[j,i,1:] =  var_range_bootstrap
        var_pdiff[i,j,1:] = var_pdiff[j,i,1:] = var_pdiff_bootstrap

In [ ]:
n_vars = var_pdiff.shape[0]
n_contexts = (n_vars)*(n_vars+1)//2

zero_pval = np.full((n_vars, n_vars), np.nan)
parent_pval = zero_pval.copy()
max_pval = zero_pval.copy()

for i in range(n_vars):
    samps = var_pdiff[i, i]
    _, p_zero = util.calc_empirical_pval(samps[1:], two_tail=False)
    zero_pval[i, i] = p_zero
    for j in range(i + 1, n_vars):
        samps = var_pdiff[i, j]
        samps_p1 = var_pdiff[i, i]
        samps_p2 = var_pdiff[j, j]
        
        if np.isnan(samps.mean()):
            continue

        _, p_zero = util.calc_empirical_pval(samps[1:], two_tail=False)
        _, p1_zero = util.calc_empirical_pval(samps_p1[1:], two_tail=False)
        _, p2_zero = util.calc_empirical_pval(samps_p2[1:], two_tail=False)
        zero_pval[i, j] = zero_pval[j, i] = p_zero

        _, p_parent1 = util.calc_empirical_pval(samps[1:]-samps_p1[0], two_tail=False)
        _, p_parent2 = util.calc_empirical_pval(samps[1:]-samps_p2[0], two_tail=False)

        parent_pval[i, j] = parent_pval[j, i] = np.maximum(p_parent1, p_parent2)